In [0]:
%run ../common/environmental_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F
df_sprints = spark.read.table(bronze_table)

In [0]:
df_sprints_dropped = df_sprints.drop('url')

In [0]:
df_sprints_final = (
    df_sprints_dropped.withColumnsRenamed({
        'constructorId' : 'constructor_id',
        'driverId' : 'driver_id',
        'raceName' : 'race_name',
        'positionText' : 'finish_position_text',
        'date' : 'race_date',
        'grid' : 'grid_position',
        'laps' : 'completed_laps',
        'number' : 'car_number',
        'position' :'finish_position'
    })
    .filter(F.col('season').isNotNull() &
            F.col('round').isNotNull() &
            F.col('constructor_id').isNotNull() &
            F.col('driver_id').isNotNull())
    .dropDuplicates(['season' , 'round' , 'constructor_id' , 'driver_id'])
    .withColumn('race_name', F.initcap(F.col('race_name')))
)

In [0]:
df_sprints_final.write.format('delta').mode('overwrite').saveAsTable(silver_table)